#Import

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql import Row
from datetime import date

#Get Data From Bronze

In [0]:
# Load product category data from Azure Data Lake Storage
df = spark.read.format("parquet")\
    .option("inferSchema",True)\
    .load("abfss://bronze@storenapatdatalake.dfs.core.windows.net/erp_PX_CAT_G1V2")
df.limit(10).display()

ID,CAT,SUBCAT,MAINTENANCE
AC_BR,Accessories,Bike Racks,Yes
AC_BS,Accessories,Bike Stands,No
AC_BC,Accessories,Bottles and Cages,No
AC_CL,Accessories,Cleaners,Yes
AC_FE,Accessories,Fenders,No
AC_HE,Accessories,Helmets,Yes
AC_HP,Accessories,Hydration Packs,No
AC_LI,Accessories,Lights,Yes
AC_LO,Accessories,Locks,Yes
AC_PA,Accessories,Panniers,No


#Data Transformation

##Trimming

In [0]:
# Remove leading and trailing spaces from string columns
for field in df.schema.fields:
    if isinstance(field.dataType,StringType):
        df = df.withColumn(field.name,trim(col(field.name)))
df.limit(10).display()

ID,CAT,SUBCAT,MAINTENANCE
AC_BR,Accessories,Bike Racks,Yes
AC_BS,Accessories,Bike Stands,No
AC_BC,Accessories,Bottles and Cages,No
AC_CL,Accessories,Cleaners,Yes
AC_FE,Accessories,Fenders,No
AC_HE,Accessories,Helmets,Yes
AC_HP,Accessories,Hydration Packs,No
AC_LI,Accessories,Lights,Yes
AC_LO,Accessories,Locks,Yes
AC_PA,Accessories,Panniers,No


##Normalize Maintenance Flag to Boolean

In [0]:
# Convert maintenance values into Boolean format
df = df.withColumn(
    "maintenance",
    when(upper(col("maintenance")) == "YES", lit(True))
     .when(upper(col("maintenance")) == "NO", lit(False))
     .otherwise(None)
)
df.limit(10).display()

ID,CAT,SUBCAT,maintenance
AC_BR,Accessories,Bike Racks,true
AC_BS,Accessories,Bike Stands,false
AC_BC,Accessories,Bottles and Cages,false
AC_CL,Accessories,Cleaners,true
AC_FE,Accessories,Fenders,false
AC_HE,Accessories,Helmets,true
AC_HP,Accessories,Hydration Packs,false
AC_LI,Accessories,Lights,true
AC_LO,Accessories,Locks,true
AC_PA,Accessories,Panniers,false


##Renaming Columns

In [0]:
# Rename source columns to descriptive names
RENAME_MAP = {
    "id": "category_id",
    "cat": "category",
    "subcat": "subcategory",
    "maintenance": "maintenance_flag"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)
df.limit(10).display()

category_id,category,subcategory,maintenance_flag
AC_BR,Accessories,Bike Racks,true
AC_BS,Accessories,Bike Stands,false
AC_BC,Accessories,Bottles and Cages,false
AC_CL,Accessories,Cleaners,true
AC_FE,Accessories,Fenders,false
AC_HE,Accessories,Helmets,true
AC_HP,Accessories,Hydration Packs,false
AC_LI,Accessories,Lights,true
AC_LO,Accessories,Locks,true
AC_PA,Accessories,Panniers,false


##Create silver record_priority to filter out duplicate id

In [0]:
# Assign a row number to each category ID
w = Window.partitionBy(df["category_id"]).orderBy(lit(1))
df = df.withColumn("record_priority",row_number().over(w))
df.limit(10).display()

category_id,category,subcategory,maintenance_flag,record_priority
AC_BC,Accessories,Bottles and Cages,false,1
AC_BR,Accessories,Bike Racks,true,1
AC_BS,Accessories,Bike Stands,false,1
AC_CL,Accessories,Cleaners,true,1
AC_FE,Accessories,Fenders,false,1
AC_HE,Accessories,Helmets,true,1
AC_HP,Accessories,Hydration Packs,false,1
AC_LI,Accessories,Lights,true,1
AC_LO,Accessories,Locks,true,1
AC_PA,Accessories,Panniers,false,1


In [0]:
# Keep one record per category ID
df = df.filter(col("record_priority") == 1)
df.limit(10).display()

category_id,category,subcategory,maintenance_flag,record_priority
AC_BC,Accessories,Bottles and Cages,false,1
AC_BR,Accessories,Bike Racks,true,1
AC_BS,Accessories,Bike Stands,false,1
AC_CL,Accessories,Cleaners,true,1
AC_FE,Accessories,Fenders,false,1
AC_HE,Accessories,Helmets,true,1
AC_HP,Accessories,Hydration Packs,false,1
AC_LI,Accessories,Lights,true,1
AC_LO,Accessories,Locks,true,1
AC_PA,Accessories,Panniers,false,1


#Data Writing

In [0]:
# Save cleaned category data in Parquet format
df.write.format("parquet").\
    mode("overwrite").\
    option("path","abfss://silver@storenapatdatalake.dfs.core.windows.net/erp_PX_CAT_G1V2")\
    .save()

In [0]:
%sql
SELECT * FROM parquet.`abfss://silver@storenapatdatalake.dfs.core.windows.net/erp_PX_CAT_G1V2` LIMIT 10

category_id,category,subcategory,maintenance_flag,record_priority
AC_BC,Accessories,Bottles and Cages,false,1
AC_BR,Accessories,Bike Racks,true,1
AC_BS,Accessories,Bike Stands,false,1
AC_CL,Accessories,Cleaners,true,1
AC_FE,Accessories,Fenders,false,1
AC_HE,Accessories,Helmets,true,1
AC_HP,Accessories,Hydration Packs,false,1
AC_LI,Accessories,Lights,true,1
AC_LO,Accessories,Locks,true,1
AC_PA,Accessories,Panniers,false,1


In [0]:
%sql
SELECT * FROM parquet.`abfss://silver@storenapatdatalake.dfs.core.windows.net/erp_PX_CAT_G1V2` WHERE record_priority > 1 ORDER BY record_priority DESC LIMIT 10

category_id,category,subcategory,maintenance_flag,record_priority
